In [1]:
import numpy as np
import pandas as pd
import time

from data_processing import DataProcessor
from model import Model
from transfer import TransferModel
from slim_config import SlimConfig

In [2]:
import scanpy as sc

Data Processing

In [3]:
gene_names_list = pd.read_csv('../../data/controls/gene_names.csv')['gene_name'].tolist()

In [4]:
processor = DataProcessor(gene_names_list)
config = SlimConfig()
# model = Model(gene_names_list, '../../data/embeddings/gene_string_embeddings.v0.3.h5', config)
model = TransferModel(gene_names_list, '../../data/embeddings/gene_string_embeddings.v0.3.h5', config, transfer_c=200)

In [5]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [6]:
train_names = [n for n in DATASETS]
Y, genes, perts = processor.build_Y(train_names, genes = "union")

Fitting the model

In [7]:
model.config.K = 20
model.config.lambda_reg = 100
measured = {n: processor.load_dataset(n).var_names for n in train_names}
model.fit(Y, genes, perts, measured)

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


Make predictions

In [8]:
pert_counts = pd.read_csv('../../data/controls/pert_counts.csv')
target_genes = pert_counts['target_gene'].tolist()

In [9]:
source_controls = processor.source_ctrls(train_names)

In [10]:
context_A_data = sc.read_h5ad('../../data/controls/context_A.h5ad')
context_B_data = sc.read_h5ad('../../data/controls/context_B.h5ad')
context_C_data = sc.read_h5ad('../../data/controls/context_C.h5ad')

In [11]:
context_dict = {'A': pd.Series(processor.context_ctrl_mean(context_A_data), index = gene_names_list),
                'B': pd.Series(processor.context_ctrl_mean(context_B_data), index = gene_names_list),
                'C': pd.Series(processor.context_ctrl_mean(context_C_data), index = gene_names_list)
                }

In [12]:
# check the weights for each context before writing anything
for ctx, ctrl in context_dict.items():
    model.set_context(ctrl, source_controls, tau=0.05, mix=0.3, panel=target_genes)
    print(ctx, dict(zip(model.source_names, model.weights.round(2))))

A {'K562': np.float64(0.14), 'RPE1': np.float64(0.1), 'HepG2': np.float64(0.1), 'Jurkat': np.float64(0.65)}
B {'K562': np.float64(0.18), 'RPE1': np.float64(0.43), 'HepG2': np.float64(0.22), 'Jurkat': np.float64(0.17)}
C {'K562': np.float64(0.18), 'RPE1': np.float64(0.36), 'HepG2': np.float64(0.31), 'Jurkat': np.float64(0.15)}


In [13]:
# one entry per context, all keyed the same way; the validation set uses the same 300 targets everywhere
contexts_and_data = {'A': context_A_data, 'B': context_B_data, 'C': context_C_data}
target_genes_by_context = {c: target_genes for c in contexts_and_data}
panel_by_context = {c: target_genes for c in contexts_and_data}  # b_c is panel-matched to these targets

In [14]:
# streams one perturbation at a time (the full file built in memory would need ~33 GB);
# set_context runs inside write_submission, once per context, right before that context's cells
t = time.time()
model.write_submission(
    'TRANSFER_slim_K20_lam100_a2_gain2.5.h5ad',
    contexts_and_data,                 # raw control cells to rescale (fully loaded, not backed)
    context_dict,                      # log1p control profiles -> similarity weights
    source_controls,
    panel_by_context,
    target_genes_by_context,
    tau=0.05,
    mix=0.3,
    knockdown_fraction=0.9,
    alpha=2.0,
    gain=2.5,                         # from fid_tuning.ipynb or fid_tuning_transfer.ipynb
    rounding="stochastic",
    cells_per_pert=400,                # required by the submission format
    rng=np.random.default_rng(0),      # reproducible cell sampling
)
print(f"written in {time.time() - t:.0f}s")

written in 309s


In [15]:
# sanity check before uploading: shape, exactly 400 cells per (context, target), integer counts
import anndata as ad

sub = ad.read_h5ad('TRANSFER_slim_K20_lam100_a2_gain2.5.h5ad', backed='r')
counts = sub.obs.groupby(['context', 'target_gene'], observed=True).size()
print(sub.shape, "expected", (3 * len(target_genes) * 400, len(gene_names_list)))
print("cells per (context, target):", counts.unique(), "| groups:", len(counts))
print("gene order matches:", list(sub.var_names) == gene_names_list)
x = sub.X[:2000].data
print("integer, non-negative:", np.allclose(x, np.round(x)) and x.min() >= 0)

(360000, 18533) expected (360000, 18533)
cells per (context, target): [400] | groups: 900
gene order matches: True
integer, non-negative: True
